-------------------------
#### Use of OpenAI embedding function

- against a pre-created dataset (with GPT vectors of dim 1536)

- MUST download the data (participants)
-----------------------------

In [3]:
#pip install wget

In [1]:
import openai
import pandas as pd
import os
#import wget
from ast import literal_eval

In [2]:
# Chroma's client library for Python
import chromadb

In [3]:
EMBEDDING_MODEL = "text-embedding-3-small"

In [4]:
# Ignore unclosed SSL socket warnings - optional in case you get these errors
import warnings

warnings.filterwarnings(action="ignore", message="unclosed", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning) 

#### Load Data

In [13]:
embeddings_url = 'https://cdn.openai.com/API/examples/data/vector_database_wikipedia_articles_embedded.zip'

# # The file is ~700 MB so this will take some time ... 3 mins
wget.download(embeddings_url, out=r'D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\chromadb')

100% [......................................................................] 698933052 / 698933052

'D:\\AI-DATASETS\\02-MISC-large\\GenAI-LLMs\\chromadb/vector_database_wikipedia_articles_embedded.zip'

In [14]:
import zipfile

In [15]:
zip_location = r'D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\chromadb\vector_database_wikipedia_articles_embedded.zip'

In [16]:
unzip_location = r'D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\chromadb\data'

In [17]:
with zipfile.ZipFile(zip_location,"r") as zip_ref:
    zip_ref.extractall(unzip_location)

... From here ...

In [5]:
# huge dataset 1.7GB
csv_location = r'D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\chromadb\data\vector_database_wikipedia_articles_embedded.csv'

In [6]:
article_df = pd.read_csv(csv_location)

In [7]:
article_df.shape

(25000, 7)

In [10]:
article_df.head()

,id,url,title,text,title_vector,content_vector,vector_id
0,1,https://simple.wikipedia.org/wiki/April,April,April is the fourth month of the year in the J...,"[0.001009464613161981, -0.020700545981526375, ...","[-0.011253940872848034, -0.013491976074874401,...",0
1,2,https://simple.wikipedia.org/wiki/August,August,August (Aug.) is the eighth month of the year ...,"[0.0009286514250561595, 0.000820168002974242, ...","[0.0003609954728744924, 0.007262262050062418, ...",1
2,6,https://simple.wikipedia.org/wiki/Art,Art,Art is a creative activity that expresses imag...,"[0.003393713850528002, 0.0061537534929811954, ...","[-0.004959689453244209, 0.015772193670272827, ...",2
3,8,https://simple.wikipedia.org/wiki/A,A,A or a is the first letter of the English alph...,"[0.0153952119871974, -0.013759135268628597, 0....","[0.024894846603274345, -0.022186409682035446, ...",3
4,9,https://simple.wikipedia.org/wiki/Air,Air,Air refers to the Earth's atmosphere. Air is a...,"[0.02224554680287838, -0.02044147066771984, -0...","[0.021524671465158463, 0.018522677943110466, -...",4


In [8]:
article_df.dtypes

id                 int64
url               object
title             object
text              object
title_vector      object
content_vector    object
vector_id          int64
dtype: object

In [9]:
# Set the maximum column width to 100 characters
pd.set_option('display.max_colwidth', 100)

In [10]:
article_df[['title',	'text']].sample(10)

,title,text
17847,Order of the Thistle,"The Most Ancient and Most Noble Order of the Thistle is an order of chivalry, associated with Sc..."
3103,Beirut,Beirut is the capital of Lebanon. It is one of the oldest continuously inhabited cities of the w...
20386,Rounders (movie),Rounders is a 1998 movie about the underground world of high-stakes poker. The movie was directe...
8190,Aaron Sorkin,"Aaron Benjamin Sorkin (born June 9, 1961) is an American screenwriter and movie director. He was..."
19856,Proboscidea,"Proboscidea (meaning ""trunked beast"") is an order containing only one familiy of living animals,..."
19289,Improper rotation,An improper rotation can be understood as an inversion followed by a proper rotation. \n\nEquiva...
14739,Dynasty (album),Dynasty is a studio album by the American hard rock/heavy metal band Kiss. It was released on Ma...
11694,Physical property,"A physical property is a property, quality or way that an object is. A physical property can alw..."
19178,Intonation (music),"Intonation, in music, can have two different meanings:\n\nIn plainchant, intonation is when a so..."
14945,Logo (programming language),The article about graphical symbols is at Logo\nLogo is a programming language that is easy to l...


In [11]:
article_df_1500 = article_df.sample(1500)

In [12]:
%%time
# Read vectors from strings back into a list
article_df_1500['title_vector']   = article_df_1500.title_vector.apply(literal_eval)
article_df_1500['content_vector'] = article_df_1500.content_vector.apply(literal_eval)

CPU times: total: 19.4 s
Wall time: 21.7 s


- Convert string representations of vectors back into Python list objects
- This is necessary because vectors saved as strings (e.g., in a CSV or database) need to be 
- converted back to lists to be used in calculations or machine learning models.

Explanation:
- literal_eval safely evaluates a string containing a Python expression (like a list) and converts it into a corresponding Python object.
- When vectors are stored in a DataFrame as strings (e.g., "[0.25, 0.89, 0.56]"), 
- we need to convert them back into actual lists (e.g., [0.25, 0.89, 0.56]) for further use.


**let us check the title and content vector dimensions**

In [13]:
len(article_df_1500.iloc[0]['title_vector']), len(article_df_1500.iloc[0]['content_vector'])

(1536, 1536)

In [14]:
# Set vector_id to be a string
article_df_1500['vector_id'] = article_df_1500['vector_id'].apply(str)

In [15]:
article_df_1500.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 1500 entries, 22317 to 14771
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id              1500 non-null   int64 
 1   url             1500 non-null   object
 2   title           1500 non-null   object
 3   text            1500 non-null   object
 4   title_vector    1500 non-null   object
 5   content_vector  1500 non-null   object
 6   vector_id       1500 non-null   object
dtypes: int64(1), object(6)
memory usage: 93.8+ KB


#### Using Chroma DB

- `Instantiate the Chroma Client`

    - We'll start by creating a client instance to interact with the Chroma database.
    
- `Create Collections for Each Class of Embedding`

    - Collections in Chroma allow you to group embeddings based on specific criteria or use cases. Each collection can store embeddings that correspond to a particular class or type, such as text embeddings, image embeddings, etc.
    
- `Query Each Collection`

    - Finally, we'll perform queries on each collection to find the most similar embeddings. This could be used for tasks such as document retrieval, semantic search, or recommendation systems.

#### Instantiate the Chroma client

Create the Chroma client. By default, Chroma is ephemeral and runs in memory. However, you can easily set up a persistent configuration which writes to disk.

In [16]:
chroma_client = chromadb.EphemeralClient() 

# Equivalent to chromadb.Client(), ephemeral.
# Uncomment for persistent client
# chroma_client = chromadb.PersistentClient()

#### Embedding fn
Chroma collections allow you to store and filter with arbitrary metadata, making it easy to query subsets of the embedded data.

Chroma is already integrated with OpenAI's embedding functions. The best way to use them is on construction of a collection, as follows

In [17]:
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

In [21]:
# Test that your OpenAI API key is correctly set as an environment variable
# Note. if you run this notebook locally, you will need to reload your terminal and the notebook for the env variables to be live.

# Note. alternatively you can set a temporary env variable like this:
# os.environ["OPENAI_API_KEY"] = 'sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

In [18]:
import openai, os

In [19]:
# Load OpenAI API key from environment variables
openai_api_key = os.getenv('OPENAI_API_KEY')

In [20]:
embedding_function = OpenAIEmbeddingFunction(
    api_key   = openai_api_key, 
    model_name= EMBEDDING_MODEL)

In [21]:
wikipedia_content_collection = chroma_client.create_collection(name='wikipedia_content', 
                                                               embedding_function=embedding_function)

wikipedia_title_collection   = chroma_client.create_collection(name='wikipedia_titles', 
                                                               embedding_function=embedding_function)

#### Populate the collections
Chroma collections allow you to populate, and filter on, whatever metadata you like. Chroma can also store the text alongside the vectors, and return everything in a single query call, when this is more convenient.

For this use-case, we'll just store the `embeddings` and `IDs`, and use these to index the original dataframe.

In [22]:
# Add the content vectors
wikipedia_content_collection.add(
    ids       = article_df_1500.vector_id.tolist(),
    embeddings= article_df_1500.content_vector.tolist(),
)

# Add the title vectors
wikipedia_title_collection.add(
    ids       = article_df_1500.vector_id.tolist(),
    embeddings= article_df_1500.title_vector.tolist(),
)

#### Search the collections
Chroma handles embedding queries for you if an embedding function is set

In [23]:
def query_collection(collection, query, max_results, dataframe):
    
    results = collection.query(query_texts= query, 
                               n_results  = max_results, 
                               include    = ['distances']) 
    
    df = pd.DataFrame({
                'id':results['ids'][0], 
                'score':results['distances'][0],
                'title': dataframe[dataframe.vector_id.isin(results['ids'][0])]['title'],
                'content': dataframe[dataframe.vector_id.isin(results['ids'][0])]['text'],
                })
    
    return df

In [24]:
title_query_result = query_collection(
    collection = wikipedia_title_collection,
    query      = "modern art in Europe",
    max_results= 10,
    dataframe  = article_df_1500
)

title_query_result.head()

,id,score,title,content
17603,21733,1.961940,ITunes,"iTunes is a media player made by Apple. It came out on January 10, 2001, at the Macworld Expo in..."
19024,9318,1.965955,Wi-Fi,Wi-Fi is a way of connecting to a computer network using radio waves instead of wires. It was i...
21733,12877,1.969124,Enrique Granados,"Enrique Granados (born Lérida, Spain, 27 July 1867; died in the English Channel, 24 March 1916)..."
24133,13438,1.969898,Psychedelic music,"Psychedelic music is a term referring to different music styles and genres, such as psychedelic ..."
12877,19024,1.971707,Daddy Yankee,"Ramón ""Raymond"" Ayala, known as Daddy Yankee (born on ) in San Juan, Puerto Rico is a Puerto Ric..."


In [25]:
content_query_result = query_collection(
    collection=wikipedia_content_collection,
    query="Famous battles in Scottish history",
    max_results=10,
    dataframe=article_df_1500
)
content_query_result.head()

,id,score,title,content
5554,7437,1.918142,479 BC,479 BC was a year in the 5th century BC.\n\nEvents \nThe Persian Wars end.\n\nDeaths\n\n Confuci...
3286,20323,1.927651,August 11,"\n\nEvents\n\nUp to 1900 \n 3114 BC The Mesoamerican Long Count calendar, used by several pre-C..."
20323,5554,1.930121,14 BC,Year 14 BC was either a common year starting on Thursday or Friday or a leap year starting on We...
7437,21441,1.938874,Order of the Bath,The Most Honourable Order of the Bath is a British order of chivalry. It was founded by George I...
21606,15097,1.953847,1991 Formula One World Championship,The 1991 Formula One season was the 42nd edition of the championship. The champion was Ayrton Se...
